# Stage 6. Extract features

## Notebook Information

**Purpose:** Extract per-object morphology, intensity, and texture features from segmented field-of-view images.

**Workflow summary:** Load configuration and metadata, open illumination-corrected FOV images and matching segmentation masks, remove edge-touching labels, compute regionprops/custom intensity features plus Hessian and structure-tensor features, then save one feature table per FOV and update processing metadata.

**Run scope:** Processes all rows in the selected metadata dataframe by default. The expected input is the Part 5 metadata containing illumination-correction and segmentation file references.

**Reproducibility:** Use the configuration/parameter cell as the source of truth for paths and run parameters. For a fresh execution, run sections in order and make sure the metadata input points to the intended Part 5 metadata file rather than a previously generated Part 6 metadata file.

### Authors

| Name | Affiliation |
|---|---|
| Alessandro Ulivi | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |
| Edwin Carreno | Scientific Software Center, Heidelberg |
| Christine Schultz | Scientific Software Center, Heidelberg |
| Name Surname | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |

## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import datetime
import os
import logging
from importlib.metadata import version
from pathlib import Path

# Third-party imports
import napari
import numpy as np
import pandas as pd
import tifffile
from omegaconf import OmegaConf
from scipy.ndimage import gaussian_filter
from skimage.transform import resize
from skimage.measure import regionprops_table

# Package imports
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.listdirNHF import listdirNHF
from acid.utils.metadata.loading import load_metadata
from acid.utils.get_defaults import default_file_name
from acid.utils.label_image_utils import exclude_label_on_edge
from acid.feature_extraction.default_regionprops import default_regionpros_props
from acid.feature_extraction.extra_regionprops import regionpros_extra_props
from acid.feature_extraction.measure_hessian_matrix import MeasureHessianMatrix
from acid.feature_extraction.measure_structure_tensor import MeasureStructureTensor

### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

In [ ]:
CONFIG_PATH_FILE = Path("./config_part6.yaml")

config = OmegaConf.load(file_=CONFIG_PATH_FILE)

#### General config parameters

In [ ]:
# project
project_name = "ACID"

# # indicate preprocessing steps
# preprocessing_steps = f"gaussian smoothing individual channels with sigma {sigma}"


# # hyperparameters dataframe name - date format
# hyperparameters_date_format = "%Y%m%d-%H%M%S"

# # hyperparameters dataframe name - savingword
# hyperparameters_savingword = "hyperparameters"

# # hyperparameters dataframe name - file suffix
# hyperparameters_file_suffix = f"part{save_file_name_separator}4extra.csv"


# # --- parameters for saving secondary information ---
# # indicate the name of the directory for saving secondary outputs -
# # this directory is used to store the hyperparameters used per each run of the pipeline
# secondary_output_directory = "secondary_output"
# exist_ok = (
#     True  # if the secondary output directory already exists, do not raise an error
# )

## 2. Input Data Preparation

### 2.1 Create output directories

In [ ]:
output_path, secondary_output_path = create_output_directories(
    output_directory=config["output"]["directory"],
    secondary_output_directory=config["output"]["secondary"]["directory"],
    enable_secondary_output=config["output"]["secondary"]["enabled"],
)

### 2.2 Load metadata dataframe

#### Config

In [ ]:
# ----- Metadata

# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part4b notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
# metadata_file_name = "20260320_ACID_metadata_part_4b.csv"  # this is a placeholder
metadata_file_name = "default"  # this is a placeholder

# metadata_df directory - indicate the full path to the directory where part1 metadata have been saved
# metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
metadata_directory = "data/proc/proc_metadata"

# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"  # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None  # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = "_"

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format = "%Y%m%d"

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True

# ----- Image processing
# indicate the channel axis position - image preprocessing is iterated over the individual channels
channel_axis = 0

# indicate the sigma of gaussian smoothing
sigma = 3


# ----- Output
# indicate the path to the directory where outputs will be saved
# output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\measurement"
output_directory = "data/proc/measurement"
save_csv_index = False

#### Old code

In [ ]:
# check if using the default metadata data frame (the most recently saved)
if (
    metadata_file_name == None
    or metadata_file_name.lower() == "default"
    or metadata_file_name == ""
):
    # import target files in the metadata_directory
    metadata_files = listdirNHF(
        metadata_directory,
        target=default_metadata_file_target,
        exclude=default_metadata_file_exclude,
    )

    # get the default metadata file
    metadata_file_name = default_file_name(
        file_list=metadata_files,
        from_file_name=metadata_from_file_name,
        directory_path=metadata_directory,
        separator=metadata_default_separator,
        date_position=metadata_default_date_position,
        date_format=metadata_default_date_format,
        reverse=metadata_default_reverse,
    )

    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df

#### New code

In [ ]:
metadata_df, metadata_file_name = load_metadata(config["measurements"]["metadata"])

## 3. Extract features

- Preprocess image and segmentation mask
- Extract features
- Save results

#### Config

In [ ]:
# indicate the path to the directory storing the input images
# fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov_proc"
fov_directory = "data/proc/fov_proc"

# indicate the path to the directory storing the segmentations
# segmentation_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\seg"
segmentation_directory = "data/proc/segmentation"

# --- parameters for updating metadata data frame ---
# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# image segmentation metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the image segmentation was done. This is used for saving the date in the
# metadata dataframe
metadata_df_meta_date_format = "%y%m%d"

metadata_df_date_clm_name = "features_extraction_date"
metadata_df_file_name_clm_name = "features_data_frame"
metadata_df_method_clm_name = "features_extraction_preprocessing"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}6.csv"

# processing metadata dataframe name - date format
metadata_date_format = "%Y%m%d"

#### Old code

In [ ]:
# from tqdm.notebook import tqdm

# # initialize lists to collect processing metadata for updating - this will be used to update the metadata df
# features_extraction_date_collection = []
# features_file_name_collection = []
# preprocessing_steps = []


# # get properties to measure
# properties = default_regionpros_props()

# # get extra properties to measure
# extra_properties = regionpros_extra_props()


# # iterate through the rows of the metadata dataframe


# for file_idx in tqdm(metadata_df.index, desc="Processing files"):
#     print("---------    ---------")

#     # ---------
#     # OPEN FIELD OF VIEW AND SEGMENTATION
#     # ---------
#     try:
#         # get the name of the field of view as ome.tif file
#         # field_of_view_file = metadata_df.loc[file_idx, illum_correct_df_file_name_clm_name]
#         field_of_view_file = metadata_df.loc[
#             file_idx, "illumination_correction_file_name"
#         ]
#         print(f"Field of view: {field_of_view_file}")
#         # form the full path to the field of view file

#         fov_path = os.path.join(fov_directory, field_of_view_file)
#         print(f"Field of view path: {fov_path}")

#         # open the field of view image as an array
#         field_of_view = tifffile.imread(fov_path)

#         print(f"working on {field_of_view_file}")

#     except:
#         print(f"can't open {field_of_view_file}, skipping image segmentation")
#         continue

#     try:
#         # get the name of the segmentation mask
#         #segmentation_file = f"{field_of_view_file.removesuffix('.ome.tif')}.tif"  # this is a placeholder
#         segmentation_file = metadata_df.loc[file_idx, 'segmentation_file_name']

#         # form the full path to the segmentation mask
#         segmentation_path = os.path.join(segmentation_directory, segmentation_file)
#         print(f"Segmentation path: {segmentation_path}")

#         # open the segmentation mask as an array
#         segmentation = tifffile.imread(segmentation_path)

#         print(f"fetched segmentation file: {segmentation_file}")

#     except:
#         print(f"can't open segmentation mask, skipping image segmentation")
#         continue

#     # ---------
#     # PREPROCESSING FIELD OF VIEW
#     # ---------
#     # move channel axis to position -1, to make next steps invariant
#     # to channel axis position and for compatibility with skimage.measure.regionprops
#     preprocessed = np.moveaxis(field_of_view, channel_axis, -1)

#     # smooth field of view along channel axis - to get rid of some noise
#     preprocessed = gaussian_filter(
#         preprocessed, sigma=sigma, axes=-1
#     )  # note: channel_axis is know to be in position -1

#     # ---------
#     # PREPROCESSING SEGMENTATION MASK
#     # ---------
#     # remove objects touching the boarder
#     label_image = exclude_label_on_edge(segmentation)

#     # ---------
#     # MEASUREMENT EXTRACTION
#     # ---------
#     # extract measurements for the nucleus and cytosol segmentations
#     regionprops_features = pd.DataFrame(
#         regionprops_table(
#             label_image,
#             intensity_image=preprocessed,
#             properties=properties,
#             extra_properties=extra_properties,
#         )
#     )
#     print(regionprops_features.shape)
#     # istantiate a hessian matrix measurer
#     hessian_measurer = MeasureHessianMatrix(preprocessed)
#     hessian_features = hessian_measurer.measure_obj_hessian_matrix_eigenval(
#         label_image=label_image, axis=-1
#     )
#     print(hessian_features.shape)
#     # istantiate a structure tensor measurer
#     structure_tensor_measurer = MeasureStructureTensor(preprocessed)
#     structure_tensor_features = (
#         structure_tensor_measurer.measure_obj_struct_tensor_eigenval(
#             label_image=label_image, axis=-1
#         )
#     )

#     print(structure_tensor_features.shape)
#     # ---------
#     # MERGE MEASUREMENT
#     # ---------
#     merged_features = regionprops_features.merge(
#         hessian_features, on="label", how="right"
#     )
#     print(merged_features.shape)
#     merged_features = merged_features.merge(
#         structure_tensor_features, on="label", how="right"
#     )
#     print(merged_features.shape)

#     # print a update
#     print("features measured")

#     # ---------
#     # SAVE THE RESULTS
#     # ---------
#     feature_save_name = f"{field_of_view_file.removesuffix('.ome.tif')}.csv"
#     merged_features.to_csv(
#         os.path.join(output_directory, feature_save_name), index=save_csv_index
#     )

#     # print a update
#     print("features saved")

#     # ---------   ---------
#     # UPDATE METADATA DATAFRAME
#     # ---------   ---------
#     features_extraction_date_collection.append(
#         datetime.datetime.now().strftime(metadata_df_meta_date_format)
#     )
#     features_file_name_collection.append(feature_save_name)
#     preprocessing_steps.append(preprocessing_steps)


# print("")
# print("Features extraction completed")

# # ---------   ---------
# # UDDATE METADATA DICTIONARY
# # ---------   ---------
# metadata_df[metadata_df_date_clm_name] = features_extraction_date_collection
# metadata_df[metadata_df_file_name_clm_name] = features_file_name_collection
# metadata_df[metadata_df_method_clm_name] = preprocessing_steps


# # ---------   ---------
# # SAVE THE FINAL METADATA FILE
# # ---------   ---------

# # save the updated metadata dataframe as a csv file
# metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
# metadata_df.to_csv(
#     os.path.join(metadata_directory, metadata_df_name), index=save_csv_index
# )

# # print progress update
# print("processing metadata saved")
# print("finished")

#### New code

In [ ]:
# FUTURE: src/acid/feature_extraction/extract_measurements.py

from datetime import datetime
from pathlib import Path

import pandas as pd
import tifffile
from scipy.ndimage import gaussian_filter
from skimage.measure import regionprops_table
from tqdm.notebook import tqdm


def load_tiff(file_path, **kwargs):
    """Load a TIFF image from disk."""
    return tifffile.imread(file_path, **kwargs)


def get_required_filename(metadata_row, column_name):
    """Return a required filename from a metadata row."""
    filename = metadata_row.get(column_name)

    if pd.isna(filename) or str(filename).strip() == "":
        raise ValueError(f"Missing filename in column {column_name!r}")

    return str(filename).strip()


def load_field_of_view(field_of_view_file, config):
    """Load one background-corrected field of view."""
    file_path = Path(config.fov_directory) / field_of_view_file

    try:
        return load_tiff(file_path)
    except Exception as error:
        raise OSError(f"Could not load field of view TIFF: {file_path}") from error


def load_segmentation_mask(segmentation_file, config):
    """Load one segmentation mask."""
    file_path = Path(config.segmentation_directory) / segmentation_file

    try:
        return load_tiff(file_path)
    except Exception as error:
        raise OSError(f"Could not load segmentation mask TIFF: {file_path}") from error


def preprocess_field_of_view(field_of_view, config):
    """Prepare image shape for skimage regionprops_table."""
    preprocessed = np.moveaxis(field_of_view, config.channel_axis, -1)

    return gaussian_filter(
        preprocessed,
        sigma=config.sigma,
        axes=-1,
    )


def preprocess_segmentation_mask(segmentation):
    """Remove segmented objects touching the image edge."""
    return exclude_label_on_edge(segmentation)


def extract_regionprops_features(
    label_image, intensity_image, properties, extra_properties
):
    """Extract standard and custom regionprops measurements."""
    return pd.DataFrame(
        regionprops_table(
            label_image,
            intensity_image=intensity_image,
            properties=properties,
            extra_properties=extra_properties,
        )
    )


def extract_hessian_features(label_image, intensity_image):
    """Extract Hessian matrix eigenvalue measurements."""
    hessian_measurer = MeasureHessianMatrix(intensity_image)

    return hessian_measurer.measure_obj_hessian_matrix_eigenval(
        label_image=label_image,
        axis=-1,
    )


def extract_structure_tensor_features(label_image, intensity_image):
    """Extract structure tensor eigenvalue measurements."""
    structure_tensor_measurer = MeasureStructureTensor(intensity_image)

    return structure_tensor_measurer.measure_obj_struct_tensor_eigenval(
        label_image=label_image,
        axis=-1,
    )


def merge_feature_tables(
    regionprops_features, hessian_features, structure_tensor_features
):
    """Merge all feature tables on object label."""
    merged_features = regionprops_features.merge(
        hessian_features,
        on="label",
        how="right",
    )

    return merged_features.merge(
        structure_tensor_features,
        on="label",
        how="right",
    )


def make_features_output_filename(field_of_view_file, config):
    """Create the feature CSV filename for one field of view."""
    stem = str(field_of_view_file).removesuffix(config.ome_suffix)

    return f"{stem}{config.output_suffix}"


def save_features_dataframe(features_df, output_filename, config):
    """Save one feature dataframe as CSV."""
    output_path = Path(config.output_directory) / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    features_df.to_csv(
        output_path,
        index=config.save_csv_index,
    )

    return output_path

In [ ]:
# FUTURE: src/acid/feature_extraction/extract_measurements.py


def make_feature_success_result(
    row_index, field_of_view_file, segmentation_file, output_file, config
):
    current_date = datetime.now().strftime(config.metadata_df_meta_date_format)

    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.metadata_df_date_clm_name: current_date,
        config.metadata_df_file_name_clm_name: output_file,
        config.metadata_df_method_clm_name: config.preprocessing_steps,
    }


def make_feature_failure_result(
    row_index,
    field_of_view_file,
    segmentation_file,
    error,
    config,
    stage=None,
):
    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
        config.metadata_df_date_clm_name: config.null_value,
        config.metadata_df_file_name_clm_name: config.null_value,
        config.metadata_df_method_clm_name: config.null_value,
    }

In [ ]:
# FUTURE: src/acid/feature_extraction/extract_measurements.py


def make_feature_success_result(
    row_index, field_of_view_file, segmentation_file, output_file, config
):
    current_date = datetime.now().strftime(config.metadata_df_meta_date_format)

    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.metadata_df_date_clm_name: current_date,
        config.metadata_df_file_name_clm_name: output_file,
        config.metadata_df_method_clm_name: config.preprocessing_steps,
    }


def make_feature_failure_result(
    row_index,
    field_of_view_file,
    segmentation_file,
    error,
    config,
    stage=None,
):
    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
        config.metadata_df_date_clm_name: config.null_value,
        config.metadata_df_file_name_clm_name: config.null_value,
        config.metadata_df_method_clm_name: config.null_value,
    }

In [ ]:
# FUTURE: src/acid/feature_extraction/extract_measurements.py


def extract_features_for_fov(
    row_index,
    metadata_row,
    config,
    properties,
    extra_properties,
):
    """Extract and save features for one metadata row."""
    print("---------    ---------")

    field_of_view_file = get_required_filename(
        metadata_row,
        config.fov_column_name,
    )

    print(f"Field of view: {field_of_view_file}")

    fov_path = Path(config.fov_directory) / field_of_view_file
    print(f"Field of view path: {fov_path}")

    field_of_view = load_field_of_view(field_of_view_file, config)
    print(f"working on {field_of_view_file}")

    segmentation_file = get_required_filename(
        metadata_row,
        config.segmentation_column_name,
    )

    segmentation_path = Path(config.segmentation_directory) / segmentation_file
    print(f"Segmentation path: {segmentation_path}")

    segmentation = load_segmentation_mask(segmentation_file, config)
    print(f"fetched segmentation file: {segmentation_file}")

    preprocessed = preprocess_field_of_view(field_of_view, config)
    # label_image = preprocess_segmentation_mask(segmentation)
    label_image = segmentation

    regionprops_features = extract_regionprops_features(
        label_image=label_image,
        intensity_image=preprocessed,
        properties=properties,
        extra_properties=extra_properties,
    )
    print(regionprops_features.shape)

    hessian_features = extract_hessian_features(
        label_image=label_image,
        intensity_image=preprocessed,
    )
    print(hessian_features.shape)

    structure_tensor_features = extract_structure_tensor_features(
        label_image=label_image,
        intensity_image=preprocessed,
    )
    print(structure_tensor_features.shape)

    merged_features = regionprops_features.merge(
        hessian_features,
        on="label",
        how="right",
    )
    print(merged_features.shape)

    merged_features = merged_features.merge(
        structure_tensor_features,
        on="label",
        how="right",
    )
    print(merged_features.shape)

    print("features measured")

    output_filename = make_features_output_filename(field_of_view_file, config)

    save_features_dataframe(
        features_df=merged_features,
        output_filename=output_filename,
        config=config,
    )

    print("features saved")

    return make_feature_success_result(
        row_index=row_index,
        field_of_view_file=field_of_view_file,
        segmentation_file=segmentation_file,
        output_file=output_filename,
        config=config,
    )


def extract_features_batch(metadata_df, config, max_rows=None):
    row_indices = metadata_df.index

    if max_rows is not None:
        row_indices = metadata_df.index[:max_rows]

    properties = default_regionpros_props()
    extra_properties = regionpros_extra_props()

    results = []

    for row_index in tqdm(row_indices, desc="Extracting features"):
        result = extract_features_for_fov(
            row_index=row_index,
            metadata_row=metadata_df.loc[row_index],
            config=config,
            properties=properties,
            extra_properties=extra_properties,
        )

        results.append(result)

    return results

In [ ]:
# FUTURE: src/acid/feature_extraction/metadata.py

FEATURE_METADATA_COLUMN_CONFIG_KEYS = (
    "metadata_df_date_clm_name",
    "metadata_df_file_name_clm_name",
    "metadata_df_method_clm_name",
)


def get_feature_metadata_columns(config):
    return [getattr(config, key) for key in FEATURE_METADATA_COLUMN_CONFIG_KEYS]


def update_metadata_with_feature_results(
    metadata_df, results, config, copy_dataframe=True
):
    if copy_dataframe:
        metadata_df = metadata_df.copy()

    if not results:
        return metadata_df

    metadata_columns = get_feature_metadata_columns(config)

    results_df = pd.DataFrame.from_records(results).set_index("row_index")

    missing_result_columns = [
        column for column in metadata_columns if column not in results_df.columns
    ]

    if missing_result_columns:
        raise KeyError(
            f"Feature extraction results are missing metadata columns: {missing_result_columns}"
        )

    updates_df = results_df[metadata_columns]

    missing_metadata_columns = [
        column for column in metadata_columns if column not in metadata_df.columns
    ]

    metadata_df = metadata_df.assign(
        **{column: pd.NA for column in missing_metadata_columns}
    )

    metadata_df = metadata_df.astype(dict.fromkeys(metadata_columns, "object"))

    metadata_df.loc[updates_df.index, metadata_columns] = updates_df.to_numpy()

    return metadata_df

In [ ]:
# FUTURE: src/acid/utils/metadata/saving.py


def build_metadata_dataframe_filename(config, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now()

    separator = config.save_file_name_separator

    return separator.join(
        [
            timestamp.strftime(config.metadata_date_format),
            config.project_name,
            config.metadata_savingword,
            config.metadata_file_suffix,
        ]
    )


def save_metadata_dataframe(
    metadata_df, metadata_config, measurement_config, timestamp=None
):
    metadata_filename = build_metadata_dataframe_filename(
        config=measurement_config,
        timestamp=timestamp,
    )

    metadata_path = Path(metadata_config.directory) / metadata_filename
    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    metadata_df.to_csv(
        metadata_path,
        index=measurement_config.save_csv_index,
    )

    return metadata_path

In [ ]:
measurement_config = config.measurements

results = extract_features_batch(
    metadata_df=metadata_df,
    config=measurement_config,
)

In [ ]:
results_df = pd.DataFrame.from_records(results)
results_df

## 4. Save results

In [ ]:
metadata_df_updated = update_metadata_with_feature_results(
    metadata_df=metadata_df,
    results=results,
    config=measurement_config,
)

metadata_df_updated.info()

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df=metadata_df_updated,
    metadata_config=config.measurements.metadata,
    measurement_config=measurement_config,
)

print("processing metadata saved")
print("finished")

metadata_path

## 4. Save analysis hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {
    "fov_directory": fov_directory,
    "segmentation_directory": segmentation_directory,
    "output_directory": output_directory,
    "metadata_directory": metadata_directory,
    "metadata_file_name": metadata_file_name,
    "channel_axis": channel_axis,
    "sigma": sigma,
    "default_metadata_file_target": default_metadata_file_target,
    "default_metadata_file_exclude": default_metadata_file_exclude,
    "metadata_from_file_name": metadata_from_file_name,
    "metadata_default_separator": metadata_default_separator,
    "metadata_default_date_position": metadata_default_date_position,
    "metadata_default_date_format": metadata_default_date_format,
    "metadata_default_reverse": metadata_default_reverse,
    "metadata_df_meta_date_format": metadata_df_meta_date_format,
    "preprocessing_steps": preprocessing_steps,
    "metadata_df_date_clm_name": metadata_df_date_clm_name,
    "metadata_df_file_name_clm_name": metadata_df_file_name_clm_name,
    "metadata_df_method_clm_name": metadata_df_method_clm_name,
    "save_file_name_separator": save_file_name_separator,
    "project_name": project_name,
    "metadata_savingword": metadata_savingword,
    "metadata_file_suffix": metadata_file_suffix,
    "metadata_date_format": metadata_date_format,
    "hyperparameters_date_format": hyperparameters_date_format,
    "hyperparameters_savingword": hyperparameters_savingword,
    "hyperparameters_file_suffix": hyperparameters_file_suffix,
    "secondary_output_directory": secondary_output_directory,
    "exist_ok": exist_ok,
}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(
    os.path.join(secondary_output_directory, hyperparameter_saving_name),
    index=save_csv_index,
)